# Taller resuelto: Árbol de Decisión — Clasificación de Vinos

Este notebook resuelve paso a paso un taller para entrenar y evaluar un árbol de decisión sobre un dataset de vinos. Está escrito en **español** y contiene EDA, entrenamiento, evaluación y visualización.

Se usan `scikit-learn`, `pandas`, `matplotlib` y `openpyxl` para exportar resultados a Excel.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

print('librerías cargadas')

## 1) Cargar datos
Cargamos el dataset de vinos disponible en `sklearn.datasets`. Convertimos a `DataFrame` para facilitar el análisis.

In [ ]:
data = load_wine()
df = pd.DataFrame(data=data.data, columns=data.feature_names)
df['target'] = data.target
df.head()


## 2) Análisis exploratorio (EDA)
Mostramos dimensiones, estadísticas y distribución de clases.

In [ ]:
print('Dimensiones:', df.shape)
print('\nDistribución de clases:')
print(df['target'].value_counts())

# Estadísticas descriptivas
summary = df.describe().T
summary['missing'] = df.isna().sum()
summary


In [ ]:
# Gráfica rápida: matriz de correlación y heatmap
corr = df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', linewidths=0.5)
plt.title('Mapa de correlación — características')
plt.show()


## 3) Preprocesamiento
Separamos features y target, normalizamos (opcional) y dividimos en train/test.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

# División
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 4) Entrenamiento
Entrenamos un `DecisionTreeClassifier` con búsqueda de hiperparámetros (`GridSearchCV`) para elegir la mejor profundidad y criterio.

In [ ]:
params = {'criterion':['gini','entropy'], 'max_depth':[None,2,3,4,5,6,7,8], 'min_samples_split':[2,4,6]}
clf = GridSearchCV(DecisionTreeClassifier(random_state=42), params, cv=5, n_jobs=-1, scoring='accuracy')
clf.fit(X_train, y_train)
print('Mejor score CV:', clf.best_score_)
print('Mejores parámetros:', clf.best_params_)

best = clf.best_estimator_


In [ ]:
y_pred = best.predict(X_test)
print('Accuracy test:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred))
print('\nMatriz de confusión:\n', confusion_matrix(y_test, y_pred))


In [ ]:
# Guardar reporte de clasificación y matriz de confusión en un Excel
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).T
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=[f'true_{i}' for i in range(cm.shape[0])], columns=[f'pred_{i}' for i in range(cm.shape[1])])

with pd.ExcelWriter('/mnt/data/resultados_arbol_vinos.xlsx') as writer:
    report_df.to_excel(writer, sheet_name='classification_report')
    cm_df.to_excel(writer, sheet_name='confusion_matrix')

print('Resultados guardados en /mnt/data/resultados_arbol_vinos.xlsx')


## 5) Visualización del árbol
Mostramos la estructura textual y la gráfica del árbol final.

In [ ]:
# Mostrar árbol en texto
arbol_text = export_text(best, feature_names=list(X.columns))
print(arbol_text[:1000])

# Plot
plt.figure(figsize=(16,10))
plot_tree(best, feature_names=X.columns, class_names=[str(c) for c in data.target_names], filled=True, rounded=True)
plt.title('Árbol de decisión final')
plt.show()


## 6) Conclusiones
- Se ha entrenado un árbol de decisión con búsqueda de hiperparámetros.
- Se guardaron el reporte y la matriz de confusión en un archivo Excel.
- Observaciones finales y posibles mejoras: poda, validación adicional, uso de RandomForest para comparar.

In [ ]:
print('Archivos generados:')
print('- Notebook: /mnt/data/Taller_Arbol_Decision_Vinos_resuelto.ipynb')
print('- Resultados (Excel): /mnt/data/resultados_arbol_vinos.xlsx')
